# Spotify 楽曲人気度予測：説明可能AI（XAI）・SHAP 分析

**Phase 2**  
**目的**: RandomForest・GradientBoosting の予測根拠を Feature Importance および SHAP で可視化・解釈する  
**データ**: Kaggle Spotify 1 Million Tracks（約116万曲、2000〜2023年）

## 0. セットアップ・データ読み込み

In [ ]:
# shap はColab環境に標準でインストール済み（0.51.0 確認済み）
%pip install japanize-matplotlib --quiet

In [ ]:
from urllib.request import urlretrieve
from zipfile import ZipFile
import os

boxurl = "https://tus.box.com/shared/static/491ie3a5kdgg7hfajivmwlou6zq2fnzj.zip"
filename = "Kaggle_Spotify_1Million_Tracks.zip"

if not os.path.exists("/content/spotify_data.csv"):
    print("データをダウンロード中...")
    urlretrieve(boxurl, filename)
    ZipFile(filename).extractall()
    print("完了")
else:
    print("データは既に存在します")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import japanize_matplotlib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance

plt.rcParams['font.family'] = 'IPAexGothic'
plt.rcParams['axes.unicode_minus'] = False

print(f"shap バージョン: {shap.__version__}")

In [ ]:
df = pd.read_csv("/content/spotify_data.csv", index_col=0)
print(f"データ形状: {df.shape}")
df.head(3)

## 1. 前処理・特徴量エンジニアリング

In [ ]:
df_clean = df.dropna(subset=['popularity']).copy()

# アーティスト出現頻度（アーティスト人気の代理変数）
artist_freq = df_clean['artist_name'].value_counts()
df_clean['artist_freq'] = df_clean['artist_name'].map(artist_freq)

# ジャンルをラベルエンコーディング
le = LabelEncoder()
df_clean['genre_enc'] = le.fit_transform(df_clean['genre'].astype(str))

FEATURE_COLS = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence',
    'tempo', 'duration_ms', 'key', 'mode', 'time_signature',
    'year', 'artist_freq', 'genre_enc'
]

X = df_clean[FEATURE_COLS]
y = df_clean['popularity']

# 訓練60% / 検証20% / テスト20%
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"訓練:   {len(X_train):,} 件")
print(f"検証:   {len(X_val):,} 件")
print(f"テスト: {len(X_test):,} 件")

## 2. モデル学習

Phase 1（`spotify_popularity_comparison.ipynb`）と同じパラメータでモデルを再学習します。

In [ ]:
print("RandomForest 学習中（約10分）...")
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
t0 = time.time()
rf.fit(X_train, y_train)
rf_time = time.time() - t0

rf_pred = rf.predict(X_test)
print(f"完了（{rf_time:.1f}秒）")
print(f"  MAE: {mean_absolute_error(y_test, rf_pred):.4f}")
print(f"  R²:  {r2_score(y_test, rf_pred):.4f}")

In [ ]:
print("HistGradientBoosting 学習中...")
gb = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_depth=8,
    min_samples_leaf=20,
    l2_regularization=0.1,
    random_state=42
)
t0 = time.time()
gb.fit(X_train, y_train)
gb_time = time.time() - t0

gb_pred = gb.predict(X_test)
print(f"完了（{gb_time:.1f}秒）")
print(f"  MAE: {mean_absolute_error(y_test, gb_pred):.4f}")
print(f"  R²:  {r2_score(y_test, gb_pred):.4f}")

## 3. Feature Importance（特徴量重要度）の比較

- **RandomForest**: `feature_importances_` を直接取得
- **GradientBoosting**: Permutation Importance（特徴量をランダムにシャッフルして精度低下を測定）

In [ ]:
# RF: feature_importances_ を直接取得
rf_imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

# GB: Permutation Importance（検証データ3,000件でサンプリング）
print("GB の Permutation Importance を計算中（1〜2分）...")
idx_perm = np.random.default_rng(42).choice(len(X_val), size=3000, replace=False)
perm_result = permutation_importance(
    gb,
    X_val.iloc[idx_perm],
    y_val.iloc[idx_perm],
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)
gb_imp = pd.Series(perm_result.importances_mean, index=FEATURE_COLS).sort_values(ascending=False)
print("完了")

print("\n=== RandomForest Feature Importance TOP10 ===")
print(rf_imp.head(10).round(4))
print("\n=== GradientBoosting Permutation Importance TOP10 ===")
print(gb_imp.head(10).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, imp, name, color in zip(
    axes,
    [rf_imp.sort_values(), gb_imp.sort_values()],
    ['RandomForest（feature_importances_）', 'GradientBoosting（Permutation Importance）'],
    ['steelblue', 'darkorange']
):
    ax.barh(imp.index, imp.values, color=color, alpha=0.8)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('重要度')

plt.suptitle('Feature Importance 比較', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. SHAP 分析

SHAP（SHapley Additive exPlanations）を用いて、各特徴量が個々の予測にどれだけ寄与しているかを定量的に分析します。

- **正のSHAP値**: その特徴量が人気度を全体平均より押し上げた
- **負のSHAP値**: その特徴量が人気度を全体平均より押し下げた

### 4-1. SHAP Explainer の構築

計算コスト削減のため、テストデータから 3,000 件をサンプリングして SHAP 値を計算します。

In [ ]:
SHAP_SAMPLE = 3000
rng = np.random.default_rng(42)
idx_shap = rng.choice(len(X_test), size=SHAP_SAMPLE, replace=False)
X_shap = X_test.iloc[idx_shap].reset_index(drop=True)
y_shap = y_test.iloc[idx_shap].reset_index(drop=True)

print(f"SHAP 計算対象: {len(X_shap):,} 件")

In [ ]:
# RF の SHAP 分析はスキップ（200本・深さ20の木構造により計算コストが高いため）
# SHAP 分析は精度が優位な GradientBoosting に絞って実施する
print("RF SHAP: スキップ（GB のみで SHAP 分析を実施）")

In [ ]:
# GradientBoosting: Explainer（自動選択）
print("GB の SHAP 値を計算中（数分かかる場合があります）...")
# 背景データとして訓練データから200件をサンプリング
bg_idx = rng.choice(len(X_train), size=200, replace=False)
X_bg = X_train.iloc[bg_idx]

explainer_gb = shap.Explainer(gb, X_bg)
shap_values_gb = explainer_gb(X_shap)
print("完了")

### 4-2. Summary Plot（全体傾向の把握）

全特徴量のSHAP値の分布を可視化します。

**読み方**:
- 縦軸: 特徴量（SHAP値の絶対値が大きい順）
- 横軸: SHAP値（右 = 人気度を押し上げ、左 = 押し下げ）
- 点の色: 特徴量の値（赤 = 高い、青 = 低い）

In [ ]:
# GradientBoosting: Summary Plot (beeswarm)
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values_gb,
    X_shap,
    feature_names=FEATURE_COLS,
    show=False
)
plt.title('GradientBoosting - SHAP Summary Plot', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary_gb.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# GradientBoosting: Summary Plot (棒グラフ)
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values_gb,
    X_shap,
    feature_names=FEATURE_COLS,
    plot_type='bar',
    show=False
)
plt.title('GradientBoosting - SHAP 平均重要度', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar_gb.png', dpi=150, bbox_inches='tight')
plt.show()

### 4-3. Summary Plot（棒グラフ版）

各特徴量の平均的な重要度を棒グラフで比較します。Feature Importanceとの違いを確認してください。

In [ ]:
# cell-23 は cell-21 に統合済みのため不要
print("Summary Plot（棒グラフ）は上のセルで出力済み")

### 4-4. Dependence Plot（特徴量の値とSHAP値の関係）

重要度上位の特徴量について、「特徴量の値」と「SHAP値（人気度への影響）」の関係を可視化します。

**読み方**:
- 横軸: 特徴量の値
- 縦軸: SHAP値（正 = 人気を押し上げ）
- 点の色: 交互作用が強い別特徴量の値

In [ ]:
# GradientBoosting の SHAP 重要度上位4特徴量を取得
shap_mean_abs_gb = np.abs(shap_values_gb.values).mean(axis=0)
top4_features = pd.Series(shap_mean_abs_gb, index=FEATURE_COLS).sort_values(ascending=False).head(4).index.tolist()
print(f"Dependence Plot 対象特徴量: {top4_features}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat in zip(axes.flatten(), top4_features):
    shap.dependence_plot(
        feat,
        shap_values_gb.values,
        X_shap,
        feature_names=FEATURE_COLS,
        ax=ax,
        show=False
    )
    ax.set_title(f'{feat} と SHAP 値の関係', fontsize=11)

plt.suptitle('GradientBoosting - Dependence Plot（重要度上位4特徴量）', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_dependence_gb.png', dpi=150, bbox_inches='tight')
plt.show()

### 4-5. Waterfall Plot（個別楽曲の予測根拠）

個別の楽曲を取り上げ、「なぜその人気度スコアが予測されたか」を特徴量ごとに分解して可視化します。

**読み方**:
- 各バーが「その特徴量が予測値をどれだけ変化させたか」を表す
- 赤 = 人気度を押し上げた特徴量
- 青 = 人気度を押し下げた特徴量

In [ ]:
# 高人気・低人気・中程度の楽曲を1件ずつ取り上げる
gb_pred_shap = gb.predict(X_shap)

idx_high = np.argmax(gb_pred_shap)         # 最も高く予測された楽曲
idx_low  = np.argmin(gb_pred_shap)         # 最も低く予測された楽曲
idx_mid  = np.argmin(np.abs(gb_pred_shap - np.median(gb_pred_shap)))  # 中央値に最も近い楽曲

targets = [
    (idx_high, f'高人気予測例（予測値: {gb_pred_shap[idx_high]:.1f}）'),
    (idx_mid,  f'中程度予測例（予測値: {gb_pred_shap[idx_mid]:.1f}）'),
    (idx_low,  f'低人気予測例（予測値: {gb_pred_shap[idx_low]:.1f}）'),
]

for idx, title in targets:
    plt.figure(figsize=(10, 5))
    shap.plots.waterfall(shap_values_gb[idx], show=False)
    plt.title(f'GradientBoosting - Waterfall Plot\n{title}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    safe_title = title.split('（')[0].replace(' ', '_')
    plt.savefig(f'shap_waterfall_{safe_title}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Feature Importance と SHAP の比較・考察

In [ ]:
# 各手法の重要度ランキングを比較する表（RF Feature Importance vs GB Permutation Importance vs GB SHAP）
shap_mean_abs_gb = np.abs(shap_values_gb.values).mean(axis=0)

comparison_df = pd.DataFrame({
    'RF_FeatureImportance_rank': rf_imp.rank(ascending=False).astype(int),
    'GB_PermImportance_rank':    gb_imp.rank(ascending=False).astype(int),
    'GB_SHAP_rank':              pd.Series(shap_mean_abs_gb, index=FEATURE_COLS).rank(ascending=False).astype(int),
}, index=FEATURE_COLS).sort_values('GB_SHAP_rank')

print("=== 各手法による特徴量重要度ランキング比較（GB_SHAP順）===")
print(comparison_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

x = np.arange(len(FEATURE_COLS))
width = 0.25
feat_order = comparison_df.index

bars = [
    ('RF Feature Importance',     rf_imp[feat_order].values,                                            'steelblue'),
    ('GB Permutation Importance',  gb_imp[feat_order].values,                                           'darkorange'),
    ('GB SHAP（平均|SHAP値|）',    shap_mean_abs_gb[[FEATURE_COLS.index(f) for f in feat_order]],      'tomato'),
]

for i, (label, vals, color) in enumerate(bars):
    normed = vals / vals.max()
    ax.bar(x + i * width, normed, width, label=label, color=color, alpha=0.8)

ax.set_xticks(x + width)
ax.set_xticklabels(feat_order, rotation=45, ha='right')
ax.set_ylabel('正規化された重要度（最大値=1）')
ax.set_title('Feature Importance vs SHAP の比較（正規化）', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('fi_vs_shap_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. まとめ

In [ ]:
print("=" * 55)
print("         Phase 2 分析まとめ")
print("=" * 55)

print("\n■ GradientBoosting SHAP 重要度 TOP5:")
top5 = pd.Series(shap_mean_abs_gb, index=FEATURE_COLS).sort_values(ascending=False).head(5)
for rank, (feat, val) in enumerate(top5.items(), 1):
    print(f"  {rank}位: {feat:<20} 平均|SHAP値| = {val:.4f}")

print("\n■ GB Permutation Importance と SHAP のランキング一致度:")
from scipy.stats import spearmanr
gb_perm_rank = gb_imp.rank(ascending=False)
gb_shap_rank = pd.Series(shap_mean_abs_gb, index=FEATURE_COLS).rank(ascending=False)
corr_gb, _ = spearmanr(gb_perm_rank, gb_shap_rank)
print(f"  GB (Permutation Importance vs SHAP): スピアマン相関 = {corr_gb:.3f}")

print("\n■ RF Feature Importance と GB SHAP のランキング一致度:")
rf_fi_rank = rf_imp.rank(ascending=False)
corr_rf_gb, _ = spearmanr(rf_fi_rank, gb_shap_rank)
print(f"  RF Feature Importance vs GB SHAP: スピアマン相関 = {corr_rf_gb:.3f}")

print("\n" + "=" * 55)